In [6]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

filePath = 'combinedWithOdds.csv'
df = pd.read_csv(filePath)

columns_to_keep = [
    'Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
    'HTHG', 'HTAG', 'HTR', 'Attendance', 'Referee', 'HS', 'AS', 'HST', 'AST',
    'HHW', 'AHW', 'HC', 'AC', 'HF', 'AF', 'HFKC', 'AFKC', 'HO', 'AO', 'HY', 'AY',
    'HR', 'AR', '1XBH', '1XBD', '1XBA', 'B365H', 'B365D', 'B365A', 'B365>2.5', 'B365<2.5', 'B365AHH', 'B365AHA', 'B365AH',
    'BFH', 'BFD', 'BFA', 'BFEH', 'BFED', 'BFEA', 'BFDH', 'BFDD', 'BFDA',
    'BMGMH', 'BMGMD', 'BMGMA', 'BVH', 'BVD', 'BVA', 'VCH', 'VCD', 'VCA',
    'BSH', 'BSD', 'BSA', 'BWH', 'BWD', 'BWA', 'CLH', 'CLD', 'CLA',
    'GBH', 'GBD', 'GBA', 'GB>2.5', 'GB<2.5', 'GBAHH', 'GBAHA', 'GBAH',
    'IWH', 'IWD', 'IWA', 'LBH', 'LBD', 'LBA', 'LBAHH', 'LBAHA', 'LBAH',
    'PSH', 'PH', 'PSD', 'PD', 'PSA', 'PA', 'P>2.5', 'P<2.5', 'PAHH', 'PAHA',
    'SOH', 'SOD', 'SOA', 'SBH', 'SBD', 'SBA', 'SJH', 'SJD', 'SJA',
    'SYH', 'SYD', 'SYA', 'WHH', 'WHD', 'WHA',
    'Bb1X2', 'BbMxH', 'BbAvH', 'BbMxD', 'BbAvD', 'BbMxA', 'BbAvA',
    'BbOU', 'BbMx>2.5', 'BbAv>2.5', 'BbMx<2.5', 'BbAv<2.5',
    'BbAH', 'BbAHh', 'BbMxAHH', 'BbAvAHH', 'BbMxAHA', 'BbAvAHA',
    'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA',
    'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5',
    'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'AHh'
]

df = df[[col for col in columns_to_keep if col in df.columns]]

drop_cols = ['Time', 'Attendance', 'HHW', 'AHW', 'HO', 'AO', 'Div']
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

def date_format_type(date_str):
    if not isinstance(date_str, str):
        return "not_a_string"
    patterns = {
        "%d/%m/%y": r"^\d{2}/\d{2}/\d{2}$",
        "%d/%m/%Y": r"^\d{2}/\d{2}/\d{4}$",
        "%Y-%m-%d": r"^\d{4}-\d{2}-\d{2}$",
        "%m-%d-%Y": r"^\d{2}-\d{2}-\d{4}$",
        "%Y/%m/%d": r"^\d{4}/\d{2}/\d{2}$",
    }
    for fmt, pat in patterns.items():
        if re.match(pat, date_str):
            return fmt
    return "unknown"

df['DateFormat'] = df['Date'].apply(date_format_type)
def parse_dates(row):
    date_str = row['Date']
    if isinstance(date_str, str):
        try:
            return pd.to_datetime(date_str, format='%d/%m/%y')
        except ValueError:
            try:
                return pd.to_datetime(date_str, format='%d/%m/%Y')
            except ValueError:
                return pd.NaT
    else:
        return pd.NaT

df['Date'] = df.apply(parse_dates, axis=1)
df = df.drop(columns=['DateFormat'], errors='ignore')
df = df[df['Date'] >= pd.Timestamp('2000-08-18')]
df = df.reset_index(drop=True)

def get_season(date):
    if pd.isnull(date):
        return np.nan
    year = date.year
    month = date.month
    if month >= 8: 
        return f"{year}-{str(year+1)[-2:]}"
    else:
        return f"{year-1}-{str(year)[-2:]}"
df['Season'] = df['Date'].apply(get_season)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek

odds_cols = [
    'IWH', 'IWD', 'IWA', 'WHH', 'WHD', 'WHA', 'B365H', 'B365D', 'B365A', 
    'B365>2.5', 'B365<2.5'
]
score_cols = [
    'FTHG', 'FTAG', 'HTHG', 'HTAG', 'HS', 'AS', 'HST', 'AST',  
    'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
]
for col in odds_cols + score_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.drop_duplicates()

core_odds = ['IWH','IWD','IWA','WHH','WHD','WHA','B365H','B365D','B365A']
df = df.dropna(subset=[c for c in core_odds if c in df.columns])
df = df.reset_index(drop=True)

df['TotalGoals'] = df['FTHG'] + df['FTAG']
df['GoalsOver2_5'] = (df['TotalGoals'] > 2.5).astype(int)
df['BTTS'] = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).astype(int)
df['Home_2plus'] = (df['FTHG'] >= 2).astype(int)
df['Away_2plus'] = (df['FTAG'] >= 2).astype(int)

df = df.sort_values('Date')
df['HomeTeam_mean_FTHG'] = (
    df.groupby('HomeTeam')['FTHG'].transform(lambda x: x.shift(1).expanding().mean())
)
df['AwayTeam_mean_FTAG'] = (
    df.groupby('AwayTeam')['FTAG'].transform(lambda x: x.shift(1).expanding().mean())
)

df['HomeTeam_str'] = df['HomeTeam']
df['AwayTeam_str'] = df['AwayTeam']
df = pd.get_dummies(df, columns=['HomeTeam', 'AwayTeam'])
df = df.rename(columns={'HomeTeam_str': 'HomeTeam', 'AwayTeam_str': 'AwayTeam'})

def add_recent_form_features(df, n_matches=5):
    base = df.copy()
    base = base.sort_values('Date')
    home_df = base[['Date', 'HomeTeam', 'FTHG', 'FTAG']].rename(
        columns={'HomeTeam': 'Team', 'FTHG': 'GoalsFor', 'FTAG': 'GoalsAgainst'})
    away_df = base[['Date', 'AwayTeam', 'FTAG', 'FTHG']].rename(
        columns={'AwayTeam': 'Team', 'FTAG': 'GoalsFor', 'FTHG': 'GoalsAgainst'})
    results = pd.concat([home_df, away_df], ignore_index=True)
    results = results.sort_values(['Team', 'Date'])
   
    def get_points(row):
        return 3 if row['GoalsFor'] > row['GoalsAgainst'] else (1 if row['GoalsFor'] == row['GoalsAgainst'] else 0)
    results['Points'] = results.apply(get_points, axis=1)
    results['RollingGF'] = results.groupby('Team')['GoalsFor'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).mean())
    results['RollingGA'] = results.groupby('Team')['GoalsAgainst'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).mean())
    results['RollingPoints'] = results.groupby('Team')['Points'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).sum())
  
    def get_form(row, team_col):
        team = row[team_col]
        date = row['Date']
        row_form = results[(results['Team'] == team) & (results['Date'] < date)].sort_values('Date').tail(1)
        if row_form.empty:
            return pd.Series([np.nan, np.nan, np.nan])
        return row_form[['RollingGF', 'RollingGA', 'RollingPoints']].values[0]
    base[['HomeRecentGF', 'HomeRecentGA', 'HomeRecentPts']] = base.apply(
        lambda row: get_form(row, 'HomeTeam'), axis=1, result_type='expand')
    base[['AwayRecentGF', 'AwayRecentGA', 'AwayRecentPts']] = base.apply(
        lambda row: get_form(row, 'AwayTeam'), axis=1, result_type='expand')
    return base

df = add_recent_form_features(df, n_matches=5)
df = df.dropna(subset=['HomeRecentGF', 'AwayRecentGF'])

df = df.sort_values('Date')
h2h_home_wins = []
team_stats = {}
home_positions = []
away_positions = []
for idx, row in df.iterrows():
    home = row['HomeTeam']
    away = row['AwayTeam']
    match_date = row['Date']
    prev_matches = df[
        (((df['HomeTeam'] == home) & (df['AwayTeam'] == away)) |
         ((df['HomeTeam'] == away) & (df['AwayTeam'] == home)))
        & (df['Date'] < match_date)
    ].sort_values('Date', ascending=False).head(5)
    home_wins = ((prev_matches['HomeTeam'] == home) & (prev_matches['FTHG'] > prev_matches['FTAG'])).sum()
    h2h_home_wins.append(home_wins)

    league_table = []
    for team, stats in team_stats.items():
        league_table.append({
            'team': team,
            'points': stats['points'],
            'gd': stats['gd'],
            'scored': stats['scored']
        })
    table_df = pd.DataFrame(league_table)
    if not table_df.empty:
        table_df = table_df.sort_values(['points', 'gd', 'scored'], ascending=[False, False, False])
        table_df['position'] = range(1, len(table_df) + 1)
        home_pos = table_df[table_df['team'] == home]['position'].values[0] if home in table_df['team'].values else len(table_df) + 1
        away_pos = table_df[table_df['team'] == away]['position'].values[0] if away in table_df['team'].values else len(table_df) + 1
    else:
        home_pos = away_pos = 1
    home_positions.append(home_pos)
    away_positions.append(away_pos)
  
    home_goals = row['FTHG']
    away_goals = row['FTAG']
    for team in [home, away]:
        if team not in team_stats:
            team_stats[team] = {'points': 0, 'gd': 0, 'scored': 0}
    if home_goals > away_goals:
        team_stats[home]['points'] += 3
    elif home_goals < away_goals:
        team_stats[away]['points'] += 3
    else:
        team_stats[home]['points'] += 1
        team_stats[away]['points'] += 1
    team_stats[home]['gd'] += home_goals - away_goals
    team_stats[away]['gd'] += away_goals - home_goals
    team_stats[home]['scored'] += home_goals
    team_stats[away]['scored'] += away_goals

df['h2h_home_wins_last5'] = h2h_home_wins
df['home_league_position'] = home_positions
df['away_league_position'] = away_positions
df['position_diff'] = df['home_league_position'] - df['away_league_position']

N = 5
df['HomePts'] = np.where(df['FTHG'] > df['FTAG'], 3, np.where(df['FTHG'] == df['FTAG'], 1, 0))
df['AwayPts'] = np.where(df['FTAG'] > df['FTHG'], 3, np.where(df['FTAG'] == df['FTHG'], 1, 0))
df['HomeRecentPts'] = (
    df.groupby('HomeTeam')['HomePts'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentPts'] = (
    df.groupby('AwayTeam')['AwayPts'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['HomeGoalDiff'] = df['FTHG'] - df['FTAG']
df['AwayGoalDiff'] = df['FTAG'] - df['FTHG']
df['HomeRecentGoalDiff'] = (
    df.groupby('HomeTeam')['HomeGoalDiff'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentGoalDiff'] = (
    df.groupby('AwayTeam')['AwayGoalDiff'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['RecentGoalDiff'] = df['HomeRecentGoalDiff'] - df['AwayRecentGoalDiff']
df['HomeRecentShotsOnTarget'] = (
    df.groupby('HomeTeam')['HST'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentShotsOnTarget'] = (
    df.groupby('AwayTeam')['AST'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['RecentShotsOnTargetDiff'] = df['HomeRecentShotsOnTarget'] - df['AwayRecentShotsOnTarget']
df['PosDiff'] = df['home_league_position'] - df['away_league_position']

for col in ['B365>2.5', 'B365<2.5']:
    median = df[col].median()
    df[f'{col}_missing'] = df[col].isna().astype(int)
    df[col] = df[col].fillna(median)

df['B365>2.5_implied_prob'] = 1 / df['B365>2.5']
df['B365<2.5_implied_prob'] = 1 / df['B365<2.5']
df['OddsMargin'] = df['B365H'] / df['B365A']
df['OU_OddsMargin'] = df['B365>2.5'] / df['B365<2.5']
df['OverUnderRatio'] = df['B365>2.5'] / df['B365<2.5']

df['Weekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)
df['EarlySeason'] = (df['Month'] <= 3).astype(int)

def rolling_ref_aggression(subdf):
    agg = subdf[['HY', 'AY', 'HR', 'AR']].shift(1).sum(axis=1)
    return agg.rolling(10, min_periods=1).mean()
df = df.sort_values('Date')
df['RefereeAggression'] = (
    df.groupby('Referee').apply(lambda subdf: rolling_ref_aggression(subdf)).reset_index(level=0, drop=True)
)

df = df.replace([np.inf, -np.inf], np.nan)
for col in df.select_dtypes(include=['number']):
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include=['object', 'category']):
    df[col] = df[col].fillna(df[col].mode()[0])

df = df.drop_duplicates().reset_index(drop=True)

df.to_csv('final_football_model_data.csv', index=False)
print('Data pipeline complete. Saved as final_football_model_data.csv')

/var/folders/tk/jl3yfcz52s9020z4_8dbfkvw0000gn/T/ipykernel_99547/3314694188.py:258: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('Referee').apply(lambda subdf: rolling_ref_aggression(subdf)).reset_index(level=0, drop=True)


Data pipeline complete. Saved as final_football_model_data.csv


In [7]:
import pandas as pd
pd.set_option('display.max_columns', None)
filePath = 'final_football_model_data.csv'
df = pd.read_csv(filePath)
df

,Date,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,1XBH,1XBD,1XBA,B365H,B365D,B365A,B365>2.5,B365<2.5,B365AHH,B365AHA,B365AH,BFH,BFD,BFA,BFEH,BFED,BFEA,VCH,VCD,VCA,BSH,BSD,BSA,BWH,BWD,BWA,GBH,GBD,GBA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,IWH,IWD,IWA,LBH,LBD,LBA,LBAHH,LBAHA,LBAH,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,SOH,SOD,SOA,SBH,SBD,SBA,SJH,SJD,SJA,SYH,SYD,SYA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,Season,Year,Month,DayOfWeek,TotalGoals,GoalsOver2_5,BTTS,Home_2plus,Away_2plus,HomeTeam_mean_FTHG,AwayTeam_mean_FTAG,HomeTeam,AwayTeam,HomeTeam_Arsenal,HomeTeam_Aston Villa,HomeTeam_Birmingham,HomeTeam_Blackburn,HomeTeam_Blackpool,HomeTeam_Bolton,HomeTeam_Bournemouth,HomeTeam_Brentford,HomeTeam_Brighton,HomeTeam_Burnley,HomeTeam_Cardiff,HomeTeam_Charlton,HomeTeam_Chelsea,HomeTeam_Crystal Palace,HomeTeam_Derby,HomeTeam_Everton,HomeTeam_Fulham,HomeTeam_Huddersfield,HomeTeam_Hull,HomeTeam_Leeds,HomeTeam_Leicester,HomeTeam_Liverpool,HomeTeam_Luton,HomeTeam_Man City,HomeTeam_Man United,HomeTeam_Middlesbrough,HomeTeam_Newcastle,HomeTeam_Norwich,HomeTeam_Nott'm Forest,HomeTeam_Portsmouth,HomeTeam_QPR,HomeTeam_Reading,HomeTeam_Sheffield United,HomeTeam_Southampton,HomeTeam_Stoke,HomeTeam_Sunderland,HomeTeam_Swansea,HomeTeam_Tottenham,HomeTeam_Watford,HomeTeam_West Brom,HomeTeam_West Ham,HomeTeam_Wigan,HomeTeam_Wolves,AwayTeam_Arsenal,AwayTeam_Aston Villa,AwayTeam_Birmingham,AwayTeam_Blackburn,AwayTeam_Blackpool,AwayTeam_Bolton,AwayTeam_Bournemouth,AwayTeam_Brentford,AwayTeam_Brighton,AwayTeam_Burnley,AwayTeam_Cardiff,AwayTeam_Charlton,AwayTeam_Chelsea,AwayTeam_Crystal Palace,AwayTeam_Derby,AwayTeam_Everton,AwayTeam_Fulham,AwayTeam_Huddersfield,AwayTeam_Hull,AwayTeam_Leeds,AwayTeam_Leicester,AwayTeam_Liverpool,AwayTeam_Luton,AwayTeam_Man City,AwayTeam_Man United,AwayTeam_Middlesbrough,AwayTeam_Newcastle,AwayTeam_Norwich,AwayTeam_Nott'm Forest,AwayTeam_Portsmouth,AwayTeam_QPR,AwayTeam_Reading,AwayTeam_Sheffield United,AwayTeam_Southampton,AwayTeam_Stoke,AwayTeam_Sunderland,AwayTeam_Swansea,AwayTeam_Tottenham,AwayTeam_Watford,AwayTeam_West Brom,AwayTeam_West Ham,AwayTeam_Wigan,AwayTeam_Wolves,HomeRecentGF,HomeRecentGA,HomeRecentPts,AwayRecentGF,AwayRecentGA,AwayRecentPts,h2h_home_wins_last5,home_league_position,away_league_position,position_diff,HomePts,AwayPts,HomeGoalDiff,AwayGoalDiff,HomeRecentGoalDiff,AwayRecentGoalDiff,RecentGoalDiff,HomeRecentShotsOnTarget,AwayRecentShotsOnTarget,RecentShotsOnTargetDiff,PosDiff,B365>2.5_missing,B365<2.5_missing,B365>2.5_implied_prob,B365<2.5_implied_prob,OddsMargin,OU_OddsMargin,OverUnderRatio,Weekend,EarlySeason,RefereeAggression
0,2002-08-27,5.0,2.0,H,3.0,0.0,H,P Durkin,10.0,8.0,8.0,6.0,1.0,1.0,9.0,9.0,3.0,2.0,0.0,0.0,NaN,NaN,NaN,1.17,5.50,13.00,1.90,1.91,1.96,1.95,-0.25,NaN,NaN,NaN,NaN,NaN,NaN,2.20,3.6,3.40,2.1,3.4,3.4,2.15,3.5,3.30,1.15,6.50,12.00,1.85,1.85,1.9,1.9,-0.25,1.20,5.00,10.00,1.20,5.00,11.0,1.95,1.9,-0.25,2.26,3.75,3.47,1.83,2.07,1.96,1.95,1.18,6.00,13.00,1.17,5.50,13.000,2.1,3.4,3.4,NaN,NaN,NaN,1.16,5.50,12.00,41.0,2.25,2.15,3.65,3.45,3.73,3.42,36.0,2.02,1.92,1.94,1.85,23.0,-0.25,1.96,1.91,2.01,1.95,2.40,3.94,3.32,2.32,3.76,3.195,1.87,2.11,1.80,2.03,1.99,1.99,1.94,1.93,-0.25,2002-03,2002,8,1,7.0,1,1,1,1,2.000000,0.000000,Arsenal,West Brom,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,2.0,0.0,8.0,0.0,1.0,5.0,0,1,1,0,3,0,3

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7901 entries, 0 to 7900
Columns: 248 entries, Date to RefereeAggression
dtypes: bool(86), float64(137), int64(18), object(7)
memory usage: 10.4+ MB


In [9]:
df.describe()

,FTHG,FTAG,HTHG,HTAG,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,1XBH,1XBD,1XBA,B365H,B365D,B365A,B365>2.5,B365<2.5,B365AHH,B365AHA,B365AH,BFH,BFD,BFA,BFEH,BFED,BFEA,VCH,VCD,VCA,BSH,BSD,BSA,BWH,BWD,BWA,GBH,GBD,GBA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,IWH,IWD,IWA,LBH,LBD,LBA,LBAHH,LBAHA,LBAH,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,SOH,SOD,SOA,SBH,SBD,SBA,SJH,SJD,SJA,SYH,SYD,SYA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,Year,Month,DayOfWeek,TotalGoals,GoalsOver2_5,BTTS,Home_2plus,Away_2plus,HomeTeam_mean_FTHG,AwayTeam_mean_FTAG,HomeRecentGF,HomeRecentGA,HomeRecentPts,AwayRecentGF,AwayRecentGA,AwayRecentPts,h2h_home_wins_last5,home_league_position,away_league_position,position_diff,HomePts,AwayPts,HomeGoalDiff,AwayGoalDiff,HomeRecentGoalDiff,AwayRecentGoalDiff,RecentGoalDiff,HomeRecentShotsOnTarget,AwayRecentShotsOnTarget,RecentShotsOnTargetDiff,PosDiff,B365>2.5_missing,B365<2.5_missing,B365>2.5_implied_prob,B365<2.5_implied_prob,OddsMargin,OU_OddsMargin,OverUnderRatio,Weekend,EarlySeason,RefereeAggression
count,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,0.0,0.0,0.0,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,0.0,0.0,0.0,0.0,0.0,0.0,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.00000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,0.0,0.0,0.0,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.00000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000,7901.000000
mean,1.530313,1.170105,0.683078,0.512720,13.676244,10.861157,6.017972,4.723326,6.031515,4.773700,11.136312,11.585369,1.458296,1.768384,0.059866,0.086065,NaN,NaN,NaN,2.775076,3.980669,4.772196,1.889144,1.944470,1.958031,1.950157,-0.256771,NaN,NaN,NaN,NaN,NaN,NaN,2.766867,3.973538,4.739819,2.243407,3.496449,3.767317,2.681850,3.881809,4.440807,2.326509,3.473945,3.919978,1.849803,1.848312,1.899168,1.901215,-0.255759,2.618861,3.773908,4.25143,2.446925,3.638196,4.151146,1.948536,1.901163,-0.257689,2.645442,4.035159,4.311268,1.835582,2.089181,1.961671,1.951172,2.118912,3.268531,3.316676,2.282016,3.438833,3.733299,2.317531,3.559504,3.942182,NaN,NaN,NaN,2.722820,3.766489,4.585504,41.963043,2.700785,2.523707,3.993179,3.732497,4.862663,4.297666,36.342995,2.008672,1.908227,2.001499,1.903005,23.089989,-0.285217,1.988491,1.924222,2.116878,2.029246,2.572168,4.059875,3.701392,2.468405,3.864663,3.501083,1.874303,2.131267,1.804599,2.048604,1.991196,1.989965,1.940215,1.930969,-0.252215,2012.978863,6.765599,4.457664,2.700418,0.511201,0.505379,0.